# Patent uniqueC Trend — yearly de-duplicated incoming citations per patent

`patent_citation_trend.parquet` resolves the citation **rows** by year (`C`, `appC` and their
examiner splits); `patent_citation.parquet` carries the de-duplicated **`uniqueC_{3,5,10,all}`** —
distinct citing patents over the union of granted and pre-grant citations — but only at the four
windows. This notebook writes the missing piece: **`uniqueC` by year since grant**, so the
de-duplicated count has a trajectory.

## Input
```
PatentView/cache/citation_edges/part_*.parquet   # (cited, citing, diff, bucket): every granted and
                                                 # application citation edge, written by patent_citation.ipynb
PatentView/output/patent_metadata.parquet        # grant_year of the cited patent
PatentView/output/patent_citation.parquet        # uniqueC_{3,5,10,all}, the check
```

## Definition
For each (cited, citing) pair the **earliest** lag over all its edges, `d = min(diff)` where
`diff = citing grant year − cited grant year ≥ 0` (both sources are dated by the citing patent's
grant year, as in `patent_citation_trend`). A citing patent is counted **once**, in the year of
its earliest citation. Then `uniqueC(t) = #{citing : d = t}` and, by construction,
`Σ_{t ≤ w} uniqueC(t) = uniqueC_w` of `patent_citation.parquet` — the check in section 3.

## Output
`PatentView/output/patent_uniqueC_trend.parquet` — `patent_id, grant_year, cite_year, yrs_since_grant, uniqueC`
(one row per patent × year with ≥ 1 new citing patent).

In [1]:
import os, sys, time, glob
import numpy as np, pandas as pd
import duckdb
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PatentView')
import pv_common as pv
EDGES  = f'{pv.CACHE}/citation_edges'
META   = pv.out('patent_metadata.parquet')
CIT    = pv.out('patent_citation.parquet')
OUT_FP = pv.out('patent_uniqueC_trend.parquet')
parts = sorted(glob.glob(f'{EDGES}/*.parquet'))
assert parts, f'{EDGES} is empty -- run patent_citation.ipynb first (it writes the edge log)'
print(f'edge log : {len(parts)} parts, {sum(os.path.getsize(p) for p in parts) / 1e9:.2f} GB')
print(f'output   : {OUT_FP}')
con = duckdb.connect()
con.execute("SET memory_limit='120GB'"); con.execute(f"SET temp_directory='{pv.CACHE}/duckdb_tmp'")
con.execute('SET preserve_insertion_order=false')

edge log : 44 parts, 1.00 GB
output   : /project/jevans/Dawoon/Science of Science/PatentView/output/patent_uniqueC_trend.parquet


## 1. Build

In [2]:
%%time
# 1. earliest lag per (cited, citing) pair -> count of new citing patents per (cited, lag)
t0 = time.time()
con.execute(f"""
COPY (
  WITH e AS (SELECT cited, citing, min(diff) AS d FROM read_parquet('{EDGES}/*.parquet') GROUP BY cited, citing),
       u AS (SELECT cited AS patent_id, d AS yrs_since_grant, count(*) AS uniqueC FROM e GROUP BY 1, 2)
  SELECT u.patent_id, CAST(m.grant_year AS INTEGER) AS grant_year,
         CAST(m.grant_year + u.yrs_since_grant AS INTEGER) AS cite_year,
         CAST(u.yrs_since_grant AS INTEGER) AS yrs_since_grant, u.uniqueC::BIGINT AS uniqueC
  FROM u JOIN read_parquet('{META}') m USING (patent_id)
  WHERE u.yrs_since_grant >= 0
  ORDER BY patent_id, yrs_since_grant
) TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
st = con.execute(f"""SELECT count(*) AS rows, count(DISTINCT patent_id) AS patents, sum(uniqueC) AS citing_pairs,
                      min(cite_year) AS y0, max(cite_year) AS y1, max(yrs_since_grant) AS max_age FROM read_parquet('{OUT_FP}')""").df().iloc[0]
print(f'[{time.time() - t0:.0f}s] WROTE {OUT_FP}  ({os.path.getsize(OUT_FP) / 1e6:.0f} MB)')
print('  ' + '  '.join(f'{k} {int(v):,}' for k, v in st.items()))

[14s] WROTE /project/jevans/Dawoon/Science of Science/PatentView/output/patent_uniqueC_trend.parquet  (110 MB)
  rows 50,546,229  patents 7,160,531  citing_pairs 161,989,008  y0 1,976  y1 2,025  max_age 49


## 2. Check against the window totals

In [3]:
%%time
# 2. The check: the yearly series summed to each window must reproduce uniqueC_w exactly
chk = con.execute(f"""
WITH s AS (
  SELECT patent_id,
         sum(uniqueC) FILTER (WHERE yrs_since_grant <= 3)  AS u3,
         sum(uniqueC) FILTER (WHERE yrs_since_grant <= 5)  AS u5,
         sum(uniqueC) FILTER (WHERE yrs_since_grant <= 10) AS u10,
         sum(uniqueC)                                      AS uall
  FROM read_parquet('{OUT_FP}') GROUP BY 1),
  j AS (SELECT c.patent_id, c.uniqueC_3, c.uniqueC_5, c.uniqueC_10, c.uniqueC_all,
               coalesce(s.u3, 0) AS u3, coalesce(s.u5, 0) AS u5, coalesce(s.u10, 0) AS u10, coalesce(s.uall, 0) AS uall
        FROM read_parquet('{CIT}') c LEFT JOIN s USING (patent_id))
SELECT count(*) AS patents,
       count(*) FILTER (WHERE uniqueC_3 <> u3)  AS bad_3,  count(*) FILTER (WHERE uniqueC_5 <> u5) AS bad_5,
       count(*) FILTER (WHERE uniqueC_10 <> u10) AS bad_10, count(*) FILTER (WHERE uniqueC_all <> uall) AS bad_all,
       sum(uniqueC_5) AS tot_uniqueC_5, sum(u5) AS tot_u5
FROM j""").df().iloc[0]
for k, v in chk.items():
    print(f'  {k:<16}{int(v):>15,}')
ok = all(chk[f'bad_{w}'] == 0 for w in ('3', '5', '10', 'all'))
print('\nyearly series reproduces uniqueC_3/5/10/all for every patent: ' + ('YES' if ok else 'NO -- inspect the mismatches'))
if not ok:
    print('  a mismatch means the edge log and patent_citation.parquet come from different runs; re-run patent_citation.ipynb '
          'section 4 and this notebook together')
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') WHERE patent_id = '10000000' ORDER BY yrs_since_grant").df())
con.close()

  patents               7,160,531
  bad_3                         0
  bad_5                         0
  bad_10                        0
  bad_all                       0
  tot_uniqueC_5        49,445,000
  tot_u5               49,445,000

yearly series reproduces uniqueC_3/5/10/all for every patent: YES


,patent_id,grant_year,cite_year,yrs_since_grant,uniqueC
0,10000000,2018,2018,0,3
1,10000000,2018,2019,1,2
2,10000000,2018,2020,2,5
3,10000000,2018,2021,3,1
4,10000000,2018,2022,4,5
5,10000000,2018,2023,5,6
6,10000000,2018,2024,6,8
7,10000000,2018,2025,7,10
